In [ ]:
import pandas as pd
import os

In [ ]:
# Configure file paths
INPUT_CSV_PATH = '/Users/utkarshumang/Desktop/igleads_no_data.csv'  
OUTPUT_CSV_PATH = 'output.csv'  

In [ ]:
# Read input CSV
df = pd.read_csv(INPUT_CSV_PATH)

print(f"Loaded {len(df)} rows")
print(f"\nInput columns: {list(df.columns)}")
df.head()

In [ ]:
def map_email(row):
    """
    Email mapping logic:
    - If Contact Email and Contact Role both have values -> Contact Email
    - If only Contact Role has value -> Contact Role
    - If both are empty -> Email
    """
    contact_email = row['Contact Email']
    contact_role = row['Contact Role']
    email = row['Email']
    
    # Check if values are not null and not empty strings
    has_contact_email = pd.notna(contact_email) and str(contact_email).strip() != ''
    has_contact_role = pd.notna(contact_role) and str(contact_role).strip() != ''
    
    if has_contact_email and has_contact_role:
        return contact_email
    elif has_contact_role and not has_contact_email:
        return contact_role
    else:
        return email

In [ ]:
def map_title(row):
    """
    Title mapping logic:
    - Contact Role if Contact Email has value
    - Otherwise Null
    """
    contact_email = row['Contact Email']
    contact_role = row['Contact Role']
    
    has_contact_email = pd.notna(contact_email) and str(contact_email).strip() != ''
    
    return contact_role if has_contact_email else None

In [ ]:
def concatenate_address(row):
    """
    Concatenate address fields with commas for Industry column
    """
    fields = ['Street', 'City', 'State', 'Postcode', 'Country']
    values = [str(row[field]) for field in fields if pd.notna(row[field]) and str(row[field]).strip() != '']
    return ', '.join(values) if values else None

In [ ]:
# Create output dataframe
output_df = pd.DataFrame()

# First, map the Email column (we'll use this value for First Name if needed)
output_df['Email'] = df.apply(map_email, axis=1)

# Now map First Name with conditional logic
def map_first_name(row):
    """
    First Name mapping logic:
    - If Contact First Name has value -> Contact First Name
    - If Contact First Name is empty AND (Contact Role OR Contact Email has value) -> use the mapped Email value
    - Otherwise -> Contact First Name (which would be empty)
    """
    contact_first_name = row['Contact First Name']
    contact_email = row['Contact Email']
    contact_role = row['Contact Role']
    
    has_contact_first_name = pd.notna(contact_first_name) and str(contact_first_name).strip() != ''
    has_contact_email = pd.notna(contact_email) and str(contact_email).strip() != ''
    has_contact_role = pd.notna(contact_role) and str(contact_role).strip() != ''
    
    if has_contact_first_name:
        return contact_first_name
    elif (has_contact_role or has_contact_email):
        # Get the already-mapped email value
        return row['_mapped_email']
    else:
        return contact_first_name

# Store mapped email temporarily so we can use it for First Name
df['_mapped_email'] = output_df['Email']

output_df['First Name'] = df.apply(map_first_name, axis=1)
output_df['Last Name'] = df['Contact Last Name']
output_df['Company Name for Emails'] = df['Business']
output_df['Website'] = df['Website']
output_df['Corporate No.'] = df['Phone']
output_df['Company Linkedin Url'] = df['Youtube']

# Conditional mappings
output_df['Title'] = df.apply(map_title, axis=1)
output_df['Industry'] = df.apply(concatenate_address, axis=1)

# No mapping columns (set to None/empty)
no_mapping_columns = [
    'Personal LinkedIn', 'Facebook Url', 'Twitter Url', 'ABOUT_US', 'EBOOK', 
    'COURSES', 'RECENT_BLOG', 'TESTIMONIALS', 'WEBINAR', 'SERVICES', 
    'PODCAST', 'SHOP', 'METADATA', 'Email 1', 'Email 1 Data Point', 
    'Subsequence 1', 'Subsequence 2', 'Subsequence 3', 'Subsequence 2 Data Point',
    'Subsequence 3', 'Subsequence 3 Data Point', 'Subsequence 4', 
    'Subsequence 4 Data Point', 'Email 2', 'Email 3'
]

for col in no_mapping_columns:
    output_df[col] = None

print("Transformation complete!")
print(f"Output shape: {output_df.shape}")

In [ ]:
# Display first few rows
output_df.head()

In [ ]:
# Check column order
print("Output columns:")
print(list(output_df.columns))

In [ ]:
# Define desired column order
desired_order = [
    'First Name', 'Last Name', 'Title', 'Company Name for Emails', 'Email',
    'Personal LinkedIn', 'Website', 'Corporate No.', 'Industry', 
    'Company Linkedin Url', 'Facebook Url', 'Twitter Url', 'ABOUT_US', 
    'EBOOK', 'COURSES', 'RECENT_BLOG', 'TESTIMONIALS', 'WEBINAR', 
    'SERVICES', 'PODCAST', 'SHOP', 'METADATA', 'Email 1', 
    'Email 1 Data Point', 'Subsequence 1', 'Subsequence 2', 'Subsequence 3',
    'Subsequence 2 Data Point', 'Subsequence 3', 'Subsequence 3 Data Point',
    'Subsequence 4', 'Subsequence 4 Data Point', 'Email 2', 'Email 3'
]

output_df = output_df[desired_order]
print("Columns reordered successfully!")

In [ ]:
# Save to CSV
output_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"✓ Output saved to: {OUTPUT_CSV_PATH}")
print(f"✓ Total rows: {len(output_df)}")
print(f"✓ Total columns: {len(output_df.columns)}")

In [ ]:
# Check for any null values in key columns
key_columns = ['First Name', 'Last Name', 'Email', 'Company Name for Emails']
print("Null value counts in key columns:")
print(output_df[key_columns].isnull().sum())

In [ ]:
# Sample some rows to verify transformations
print("Sample rows:")
output_df[['First Name', 'Last Name', 'Title', 'Email', 'Industry']].sample(min(5, len(output_df)))